In [23]:
from langgraph.graph import StateGraph, START , END 
from langchain_groq import ChatGroq
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [24]:
load_dotenv()

True

In [25]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [26]:
class EvaluationSchema(BaseModel):
    feedback:str = Field(description='Detailed eedback for the essay ')
    score:int = Field(description='Score out of 10' , ge = 0, le= 10 )
    

In [27]:
structured_llm = llm.with_structured_output(EvaluationSchema)

In [28]:
essay = """India is positioning itself to become a major global AI power—not just a consumer of AI tools, but a developer of AI models, an application hub, a source of technical talent, and a contributor to global AI governance.

As of October 2026, India's strongest opportunity is to use its large technology workforce, digital infrastructure, multilingual population, and industry expertise to build affordable, scalable AI solutions for India and the world. The Economic Survey 2025–26 emphasizes an application-driven approach rather than competing solely with the US and China on the largest frontier models."""

In [29]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'

structured_llm.invoke(prompt)

EvaluationSchema(feedback='The essay is concise and clear, with a strong opening that sets the context and a logical progression of ideas. The language is formal and appropriate for an analytical piece, and the sentences are well-structured. However, the essay could benefit from more specific examples or data to support the claims, and a concluding sentence would help to reinforce the main point. Additionally, a few minor grammatical tweaks—such as adding a comma after "India" in the first sentence and ensuring consistent tense usage—would polish the piece further.', score=8)

In [30]:
class UPSCState(TypedDict):
    essay : str
    language_feedback: str  
    analysis_feedback: str 
    clarity_feedback: str 

    overall_feedback: str 
    individual_scores: Annotated[list[int], operator.add]
    avg_score : float

In [44]:
def evaluate_language(state: UPSCState):
    essay= state['essay']
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_llm.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]} 

def evaluate_analysis(state: UPSCState):
    essay= state['essay']
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_llm.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]} 

def evaluate_clarity(state: UPSCState):
    essay= state['essay']
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = structured_llm.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]} 

def final_evaluation(state:UPSCState):
    prompt = f'based on the following feedbacks create a summarised feedback \n language feedback - {state["language_feedback"] } \n analysis feedback - {state['analysis_feedback']} \n clarity feedback - {state['clarity_feedback']}'
    output = llm.invoke(prompt).content

    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback':output , 'avg_score': avg_score}


In [45]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')
graph.add_edge(START, 'evaluate_language')

graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()



In [46]:
initial_state = {'essay': essay}
final_state = workflow.invoke(initial_state)

In [48]:
print(final_state['avg_score'])

7.333333333333333
